# 03 — PU Ablation (Faster R-CNN with label masking)

Same as notebook 02 but with `pu_keep_prob = 0.6`: 40% of ground-truth boxes
are randomly dropped **from the labels** (they remain in the image) —
simulating the positive-unlabeled regime of the challenge. Compare its mAP
against notebook 02's baseline for the paper's ablation table.

In [ ]:
import glob, sys
from pathlib import Path
import torch
from torch.utils.data import DataLoader

from src.data.coco_dataset import CocoPuDataset, collate_fn
from src.models.detector import build_model
from src.training.utils import MetricLogger, set_seed, train_transforms

ANN = Path(glob.glob('/kaggle/input/**/train_dataset.json', recursive=True)[0]).parent
IMG = ANN.parent / 'images'
if not IMG.exists():
    IMG = ANN / 'images'
SRC = glob.glob('/kaggle/input/**/src', recursive=True)
if SRC: sys.path.insert(0, str(Path(SRC[0]).parent))
else: sys.path.insert(0, '..')

CFG = dict(batch_size=4, epochs=12, lr=0.005, weight_decay=5e-4,
           milestones=[8, 11], num_workers=2, pu_keep_prob=0.6, seed=42,
           tag='pu06')
set_seed(CFG['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '| pu_keep_prob:', CFG['pu_keep_prob'])

In [ ]:
ds = CocoPuDataset(IMG, ANN / 'train_dataset.json',
                   transforms=train_transforms(), pu_keep_prob=CFG['pu_keep_prob'])
loader = DataLoader(ds, batch_size=CFG['batch_size'], shuffle=True,
                    num_workers=CFG['num_workers'], collate_fn=collate_fn, pin_memory=True)
model = build_model('faster_rcnn', num_classes=33).to(device)
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                            lr=CFG['lr'], momentum=0.9, weight_decay=CFG['weight_decay'])
sched = torch.optim.lr_scheduler.MultiStepLR(optimizer, CFG['milestones'], gamma=0.1)

for epoch in range(1, CFG['epochs'] + 1):
    model.train(); logger = MetricLogger()
    for images, targets in loader:
        images = [i.to(device) for i in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        optimizer.zero_grad()
        losses = model(images, targets)
        loss = sum(losses.values())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        optimizer.step()
        logger.update(total_loss=loss.item())
    sched.step()
    print(f"Epoch {epoch} [{CFG['tag']}]: {logger}")
    torch.save({'model': model.state_dict(), 'epoch': epoch},
               f"/kaggle/working/model_{CFG['tag']}_{epoch:03d}.pth")
print('Done.')